In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.2 MB/s eta 0:00:00


In [13]:
# light version - improved v5
import os
import glob
import cv2
import pandas as pd
from ultralytics import YOLO
from google.colab import drive

# =========================
# 1. Mount Google Drive
# =========================
drive.mount('/content/drive')

# =========================
# 2. Paths
# =========================
SEQUENCE_PATH = "/content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI/data_tracking_image_2/training/image_02/0001"

OUTPUT_DIR = "/content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI_results_presentation2"
ANNOTATED_FRAMES_DIR = os.path.join(OUTPUT_DIR, "annotated_frames")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ANNOTATED_FRAMES_DIR, exist_ok=True)

# =========================
# 3. YOLO settings
# =========================
MODEL_PATH = "yolov8s.pt"
IMG_SIZE   = 640

# --- Confidence ---
PERSON_CONF_THRESHOLD = 0.65   # ισορροπία μεταξύ FP και FN
CAR_CONF_THRESHOLD    = 0.40

TARGET_CLASSES = {"person", "car"}

# --- Person size filters ---
PERSON_MIN_WIDTH  = 12
PERSON_MIN_HEIGHT = 45     # καλύπτει και μακρινούς pedestrians
PERSON_MAX_WIDTH  = 120
PERSON_MAX_HEIGHT = 350
MIN_PERSON_AREA   = 600    # pixels²

# --- Person aspect ratio (h/w) ---
PERSON_MIN_ASPECT = 1.3    # χαλαρότερο για μερικώς ορατούς
PERSON_MAX_ASPECT = 6.0

# --- Car size / aspect ---
CAR_MIN_WIDTH  = 20
CAR_MIN_HEIGHT = 15
CAR_MAX_ASPECT = 1.8

# --- Sky cutoff: πάνω 40% του frame → δεν υπάρχουν persons ---
SKY_CUTOFF_RATIO = 0.40

# --- NMS IoU thresholds ---
NMS_IOU_PERSON = 0.40
NMS_IOU_CAR    = 0.45

# --- Person–Car overlap: αυστηρότερο για να κόβει FP ---
PERSON_CAR_IOP_THRESH = 0.20   # IoP = intersection / person_area

SAVE_EVERY_N_FRAMES = 20

# YOLO COCO class ids: person=0, car=2
YOLO_CLASS_IDS = [0, 2]

# =========================
# 4. Helper functions
# =========================

def apply_nms(candidates, iou_threshold=0.40):
    """
    Greedy NMS: κρατά το box με την υψηλότερη confidence,
    αφαιρεί όσα overlap υπερβαίνουν το iou_threshold.
    """
    if not candidates:
        return []
    candidates.sort(key=lambda x: x["conf"], reverse=True)
    kept = []
    for candidate in candidates:
        dominated = False
        for kept_box in kept:
            x_left   = max(candidate["x1"], kept_box["x1"])
            y_top    = max(candidate["y1"], kept_box["y1"])
            x_right  = min(candidate["x2"], kept_box["x2"])
            y_bottom = min(candidate["y2"], kept_box["y2"])
            inter = max(0.0, x_right - x_left) * max(0.0, y_bottom - y_top)
            area1 = (candidate["x2"] - candidate["x1"]) * (candidate["y2"] - candidate["y1"])
            area2 = (kept_box["x2"]  - kept_box["x1"])  * (kept_box["y2"]  - kept_box["y1"])
            union = area1 + area2 - inter
            if union > 0 and (inter / union) > iou_threshold:
                dominated = True
                break
        if not dominated:
            kept.append(candidate)
    return kept


def remove_person_car_overlaps(person_cands, car_cands, iop_thresh=0.20):
    """
    Αφαιρεί persons που overlap σημαντικά με car boxes.
    Χρησιμοποιεί IoP (intersection / person_area) αντί IoU
    γιατί ο άνθρωπος είναι πολύ μικρότερος από το car box.
    """
    filtered = []
    for p in person_cands:
        overlaps_car = False
        area_p = (p["x2"] - p["x1"]) * (p["y2"] - p["y1"])
        if area_p <= 0:
            continue
        for c in car_cands:
            x_left   = max(p["x1"], c["x1"])
            y_top    = max(p["y1"], c["y1"])
            x_right  = min(p["x2"], c["x2"])
            y_bottom = min(p["y2"], c["y2"])
            inter = max(0.0, x_right - x_left) * max(0.0, y_bottom - y_top)
            if (inter / area_p) > iop_thresh:
                overlaps_car = True
                break
        if not overlaps_car:
            filtered.append(p)
    return filtered


# =========================
# 5. Load model
# =========================
model = YOLO(MODEL_PATH)

# =========================
# 6. Load KITTI frames
# =========================
frame_paths = sorted(glob.glob(os.path.join(SEQUENCE_PATH, "*.png")))

if len(frame_paths) == 0:
    raise ValueError(f"Δεν βρέθηκαν frames στο folder: {SEQUENCE_PATH}")

print(f"Found {len(frame_paths)} frames.")

# =========================
# 7. Process frames
# =========================
results_list = []
boxes_list   = []

for frame_idx, frame_path in enumerate(frame_paths):
    frame = cv2.imread(frame_path)

    if frame is None:
        print(f"Skipped unreadable frame: {frame_path}")
        continue

    output       = frame.copy()
    frame_height = frame.shape[0]
    sky_cutoff   = int(frame_height * SKY_CUTOFF_RATIO)

    # YOLO inference
    detections = model(
        frame,
        imgsz=IMG_SIZE,
        verbose=False,
        classes=YOLO_CLASS_IDS
    )

    person_candidates = []
    car_candidates    = []

    for r in detections:
        for box in r.boxes:
            cls   = int(box.cls[0])
            conf  = float(box.conf[0])
            label = model.names[cls]

            if label not in TARGET_CLASSES:
                continue

            x1, y1, x2, y2 = map(float, box.xyxy[0])
            w  = x2 - x1
            h  = y2 - y1
            ar = h / w if w > 0 else 0

            entry = {
                "conf": conf, "label": label,
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                "w": w, "h": h, "aspect_ratio": ar
            }

            # -----------------------------------------------
            # PERSON filters
            # -----------------------------------------------
            if label == "person":
                if conf < PERSON_CONF_THRESHOLD:   continue
                if y2 < sky_cutoff:                continue  # sky cutoff
                if w < PERSON_MIN_WIDTH:           continue
                if h < PERSON_MIN_HEIGHT:          continue
                if w > PERSON_MAX_WIDTH:           continue
                if h > PERSON_MAX_HEIGHT:          continue
                if (w * h) < MIN_PERSON_AREA:      continue  # area filter
                if ar < PERSON_MIN_ASPECT:         continue
                if ar > PERSON_MAX_ASPECT:         continue
                person_candidates.append(entry)

            # -----------------------------------------------
            # CAR filters
            # -----------------------------------------------
            elif label == "car":
                if conf < CAR_CONF_THRESHOLD:      continue
                if w < CAR_MIN_WIDTH:              continue
                if h < CAR_MIN_HEIGHT:             continue
                if ar > CAR_MAX_ASPECT:            continue
                car_candidates.append(entry)

    # -----------------------------------------------
    # NMS ξεχωριστά για person / car
    # -----------------------------------------------
    person_candidates = apply_nms(person_candidates, iou_threshold=NMS_IOU_PERSON)
    car_candidates    = apply_nms(car_candidates,    iou_threshold=NMS_IOU_CAR)

    # -----------------------------------------------
    # Αφαίρεσε persons που overlap με cars (FP)
    # -----------------------------------------------
    person_candidates = remove_person_car_overlaps(
        person_candidates, car_candidates, iop_thresh=PERSON_CAR_IOP_THRESH
    )

    person_count = len(person_candidates)
    car_count    = len(car_candidates)
    total_count  = person_count + car_count

    # -----------------------------------------------
    # Σχεδίαση + αποθήκευση boxes
    # -----------------------------------------------
    for det in person_candidates:
        cv2.rectangle(
            output,
            (int(det["x1"]), int(det["y1"])),
            (int(det["x2"]), int(det["y2"])),
            (0, 255, 0), 2
        )
        cv2.putText(
            output,
            f"person {det['conf']:.2f}",
            (int(det["x1"]), max(int(det["y1"]) - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2
        )
        boxes_list.append({
            "frame_index": frame_idx,
            "frame_name":  os.path.basename(frame_path),
            "pred_label":  "person",
            "confidence":  det["conf"],
            "x1": det["x1"], "y1": det["y1"],
            "x2": det["x2"], "y2": det["y2"],
            "width": det["w"], "height": det["h"],
            "aspect_ratio": det["aspect_ratio"]
        })

    for det in car_candidates:
        cv2.rectangle(
            output,
            (int(det["x1"]), int(det["y1"])),
            (int(det["x2"]), int(det["y2"])),
            (255, 0, 0), 2
        )
        cv2.putText(
            output,
            f"car {det['conf']:.2f}",
            (int(det["x1"]), max(int(det["y1"]) - 10, 20)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2
        )
        boxes_list.append({
            "frame_index": frame_idx,
            "frame_name":  os.path.basename(frame_path),
            "pred_label":  "car",
            "confidence":  det["conf"],
            "x1": det["x1"], "y1": det["y1"],
            "x2": det["x2"], "y2": det["y2"],
            "width": det["w"], "height": det["h"],
            "aspect_ratio": det["aspect_ratio"]
        })

    # -----------------------------------------------
    # Overlay counts
    # -----------------------------------------------
    cv2.putText(output, f"Persons: {person_count}", (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    cv2.putText(output, f"Cars: {car_count}",       (20, 70),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
    cv2.putText(output, f"Total: {total_count}",    (20, 105),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)

    # -----------------------------------------------
    # Αποθήκευση annotated frame
    # -----------------------------------------------
    if frame_idx % SAVE_EVERY_N_FRAMES == 0:
        save_name = os.path.join(ANNOTATED_FRAMES_DIR, f"annotated_{frame_idx:04d}.png")
        cv2.imwrite(save_name, output)

    results_list.append({
        "frame_index":  frame_idx,
        "frame_name":   os.path.basename(frame_path),
        "person_count": person_count,
        "car_count":    car_count,
        "total_count":  total_count
    })

    if frame_idx % 50 == 0:
        print(f"Processed frame {frame_idx}/{len(frame_paths)}")

# =========================
# 8. Save counts CSV
# =========================
df       = pd.DataFrame(results_list)
csv_path = os.path.join(OUTPUT_DIR, "kitti_detection_counts.csv")
df.to_csv(csv_path, index=False)

# =========================
# 9. Save boxes CSV
# =========================
boxes_df       = pd.DataFrame(boxes_list)
boxes_csv_path = os.path.join(OUTPUT_DIR, "kitti_pred_boxes.csv")
boxes_df.to_csv(boxes_csv_path, index=False)

# =========================
# 10. Basic statistics
# =========================
avg_persons = df["person_count"].mean()
avg_cars    = df["car_count"].mean()
avg_total   = df["total_count"].mean()

max_persons = df["person_count"].max()
max_cars    = df["car_count"].max()
max_total   = df["total_count"].max()

print("\nProcessing finished!")
print(f"CSV saved to:               {csv_path}")
print(f"Predicted boxes CSV saved:  {boxes_csv_path}")
print(f"Annotated frames saved to:  {ANNOTATED_FRAMES_DIR}")

print("\nBasic Statistics:")
print(f"Average persons per frame:        {avg_persons:.2f}")
print(f"Average cars per frame:           {avg_cars:.2f}")
print(f"Average total objects per frame:  {avg_total:.2f}")
print(f"Maximum persons in a frame:       {max_persons}")
print(f"Maximum cars in a frame:          {max_cars}")
print(f"Maximum total objects in a frame: {max_total}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 447 frames.
Processed frame 0/447
Processed frame 50/447
Processed frame 100/447
Processed frame 150/447
Processed frame 200/447
Processed frame 250/447
Processed frame 300/447
Processed frame 350/447
Processed frame 400/447

Processing finished!
CSV saved to:               /content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI_results_presentation2/kitti_detection_counts.csv
Predicted boxes CSV saved:  /content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI_results_presentation2/kitti_pred_boxes.csv
Annotated frames saved to:  /content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI_results_presentation2/annotated_frames

Basic Statistics:
Average persons per frame:        0.06
Average cars per frame:           6.09
Average total objects per frame:  6.15
Maximum persons in a frame:       2
Maximum cars in a frame:         

In [14]:
# evaluation - fixed v2
import os
import pandas as pd
from google.colab import drive

# =====================================================
# 1. Mount Google Drive
# =====================================================
drive.mount('/content/drive')

# =====================================================
# 2. Paths
# =====================================================
PREDICTION_COUNTS_CSV = "/content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI_results_presentation2/kitti_detection_counts.csv"
PREDICTION_BOXES_CSV  = "/content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI_results_presentation2/kitti_pred_boxes.csv"
LABEL_FILE            = "/content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI/data_tracking_label_2/training/label_02/0001.txt"

OUTPUT_EVAL_CSV    = "/content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI_results_presentation2/kitti_detection_eval_iou.csv"
OUTPUT_SUMMARY_CSV = "/content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI_results_presentation2/kitti_detection_eval_summary.csv"

# =====================================================
# 3. Settings
# =====================================================
IOU_THRESHOLD = 0.50

# FIX 1: Van + Truck → αντιστοιχούν σε "car" για το YOLO
TARGET_GT_CLASSES = ["Pedestrian", "Car", "Van", "Truck"]

# FIX 3: Φιλτράρισμα occluded/truncated GT
MAX_TRUNCATED = 0.5   # 0=fully visible, 1=fully truncated
MAX_OCCLUDED  = 2     # 0=fully visible, 1=partly, 2=largely, 3=fully occluded

# =====================================================
# 4. Helper: IoU
# =====================================================
def compute_iou(box1, box2):
    x_left   = max(box1[0], box2[0])
    y_top    = max(box1[1], box2[1])
    x_right  = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    inter_w = max(0.0, x_right  - x_left)
    inter_h = max(0.0, y_bottom - y_top)
    inter_area = inter_w * inter_h

    area1 = max(0.0, box1[2] - box1[0]) * max(0.0, box1[3] - box1[1])
    area2 = max(0.0, box2[2] - box2[0]) * max(0.0, box2[3] - box2[1])
    union_area = area1 + area2 - inter_area

    if union_area <= 0:
        return 0.0
    return inter_area / union_area

# =====================================================
# 5. GT label mapping
# =====================================================
def map_gt_label(gt_type):
    if gt_type == "Pedestrian":
        return "person"
    elif gt_type in ["Car", "Van", "Truck"]:   # FIX 1
        return "car"
    return None

# =====================================================
# 6. Load predictions
# =====================================================
pred_counts_df = pd.read_csv(PREDICTION_COUNTS_CSV)
pred_boxes_df  = pd.read_csv(PREDICTION_BOXES_CSV)

# =====================================================
# 7. Load KITTI ground truth labels
# =====================================================
col_names = [
    "frame", "track_id", "type", "truncated", "occluded", "alpha",
    "bbox_left", "bbox_top", "bbox_right", "bbox_bottom",
    "dim_h", "dim_w", "dim_l", "loc_x", "loc_y", "loc_z", "rotation_y"
]

gt_df_full = pd.read_csv(LABEL_FILE, sep=" ", header=None, names=col_names)

# FIX 2: Κράτα DontCare ξεχωριστά πριν φιλτράρεις
dontcare_df = gt_df_full[gt_df_full["type"] == "DontCare"].copy()

# FIX 1: Κράτα Van/Truck/Car/Pedestrian
gt_df = gt_df_full[gt_df_full["type"].isin(TARGET_GT_CLASSES)].copy()

# FIX 3: Φιλτράρισμα occluded / truncated
gt_df = gt_df[
    (gt_df["truncated"] <= MAX_TRUNCATED) &
    (gt_df["occluded"]  <= MAX_OCCLUDED)
].copy()

print(f"Total GT boxes after filtering: {len(gt_df)}")
print(f"GT class distribution:\n{gt_df['type'].value_counts()}\n")

# =====================================================
# 8. Helper: DontCare check  (FIX 2)
# =====================================================
def is_in_dontcare(pred_box, frame_dontcare_df, iou_thresh=0.50):
    """Επιστρέφει True αν το pred_box overlap με DontCare region."""
    for _, dc in frame_dontcare_df.iterrows():
        dc_box = [dc["bbox_left"], dc["bbox_top"], dc["bbox_right"], dc["bbox_bottom"]]
        if compute_iou(pred_box, dc_box) > iou_thresh:
            return True
    return False

# =====================================================
# 9. Evaluate frame-by-frame
# =====================================================
all_frames = sorted(pred_counts_df["frame_index"].unique())

eval_rows = []

global_tp = global_fp = global_fn = 0
global_tp_person = global_fp_person = global_fn_person = 0
global_tp_car    = global_fp_car    = global_fn_car    = 0

for frame_idx in all_frames:

    # --- prediction boxes ---
    frame_pred_df = pred_boxes_df[pred_boxes_df["frame_index"] == frame_idx].copy()
    pred_boxes = []
    for _, row in frame_pred_df.iterrows():
        pred_boxes.append({
            "label": row["pred_label"],
            "bbox":  [row["x1"], row["y1"], row["x2"], row["y2"]]
        })

    # --- FIX 2: φιλτράρισμα DontCare ---
    frame_dc = dontcare_df[dontcare_df["frame"] == frame_idx]
    if not frame_dc.empty:
        pred_boxes = [
            p for p in pred_boxes
            if not is_in_dontcare(p["bbox"], frame_dc, iou_thresh=0.50)
        ]

    # --- GT boxes ---
    frame_gt_df = gt_df[gt_df["frame"] == frame_idx].copy()
    gt_boxes = []
    for _, row in frame_gt_df.iterrows():
        mapped = map_gt_label(row["type"])
        if mapped is None:
            continue
        gt_boxes.append({
            "label": mapped,
            "bbox":  [row["bbox_left"], row["bbox_top"], row["bbox_right"], row["bbox_bottom"]]
        })

    # -----------------------------------------------
    # Generic matching function
    # -----------------------------------------------
    def match_boxes(preds, gts):
        matched_gt = set()
        tp = 0
        for pred in preds:
            best_iou    = 0.0
            best_gt_idx = -1
            for gt_idx, gt in enumerate(gts):
                if gt_idx in matched_gt:
                    continue
                if pred["label"] != gt["label"]:
                    continue
                iou = compute_iou(pred["bbox"], gt["bbox"])
                if iou > best_iou:
                    best_iou    = iou
                    best_gt_idx = gt_idx
            if best_iou >= IOU_THRESHOLD and best_gt_idx != -1:
                tp += 1
                matched_gt.add(best_gt_idx)
        fp = len(preds) - tp
        fn = len(gts)   - tp
        return tp, fp, fn

    def safe_metrics(tp, fp, fn):
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) > 0 else 0.0)
        return precision, recall, f1

    # --- overall ---
    tp, fp, fn          = match_boxes(pred_boxes, gt_boxes)
    precision, recall, f1 = safe_metrics(tp, fp, fn)

    # --- person ---
    person_preds = [p for p in pred_boxes if p["label"] == "person"]
    person_gts   = [g for g in gt_boxes   if g["label"] == "person"]
    tp_p, fp_p, fn_p      = match_boxes(person_preds, person_gts)
    prec_p, rec_p, f1_p   = safe_metrics(tp_p, fp_p, fn_p)

    # --- car ---
    car_preds  = [p for p in pred_boxes if p["label"] == "car"]
    car_gts    = [g for g in gt_boxes   if g["label"] == "car"]
    tp_c, fp_c, fn_c      = match_boxes(car_preds, car_gts)
    prec_c, rec_c, f1_c   = safe_metrics(tp_c, fp_c, fn_c)

    eval_rows.append({
        "frame_index":    frame_idx,
        "num_pred_boxes": len(pred_boxes),
        "num_gt_boxes":   len(gt_boxes),

        "TP": tp, "FP": fp, "FN": fn,
        "precision": precision, "recall": recall, "f1_score": f1,

        "TP_person": tp_p, "FP_person": fp_p, "FN_person": fn_p,
        "precision_person": prec_p, "recall_person": rec_p, "f1_person": f1_p,

        "TP_car": tp_c, "FP_car": fp_c, "FN_car": fn_c,
        "precision_car": prec_c, "recall_car": rec_c, "f1_car": f1_c
    })

    global_tp += tp;  global_fp += fp;  global_fn += fn
    global_tp_person += tp_p; global_fp_person += fp_p; global_fn_person += fn_p
    global_tp_car    += tp_c; global_fp_car    += fp_c; global_fn_car    += fn_c

# =====================================================
# 10. Save frame-by-frame CSV
# =====================================================
eval_df = pd.DataFrame(eval_rows)
eval_df.to_csv(OUTPUT_EVAL_CSV, index=False)

# =====================================================
# 11. Global metrics
# =====================================================
gp, gr, gf        = safe_metrics(global_tp,        global_fp,        global_fn)
gp_p, gr_p, gf_p  = safe_metrics(global_tp_person, global_fp_person, global_fn_person)
gp_c, gr_c, gf_c  = safe_metrics(global_tp_car,    global_fp_car,    global_fn_car)

summary_df = pd.DataFrame([{
    "iou_threshold": IOU_THRESHOLD,

    "global_TP": global_tp,         "global_FP": global_fp,         "global_FN": global_fn,
    "global_precision": gp,         "global_recall": gr,            "global_f1_score": gf,

    "global_TP_person": global_tp_person, "global_FP_person": global_fp_person, "global_FN_person": global_fn_person,
    "global_precision_person": gp_p,      "global_recall_person": gr_p,         "global_f1_person": gf_p,

    "global_TP_car": global_tp_car,       "global_FP_car": global_fp_car,       "global_FN_car": global_fn_car,
    "global_precision_car": gp_c,         "global_recall_car": gr_c,            "global_f1_car": gf_c,

    "avg_frame_precision":        eval_df["precision"].mean(),
    "avg_frame_recall":           eval_df["recall"].mean(),
    "avg_frame_f1":               eval_df["f1_score"].mean(),

    "avg_frame_precision_person": eval_df["precision_person"].mean(),
    "avg_frame_recall_person":    eval_df["recall_person"].mean(),
    "avg_frame_f1_person":        eval_df["f1_person"].mean(),

    "avg_frame_precision_car":    eval_df["precision_car"].mean(),
    "avg_frame_recall_car":       eval_df["recall_car"].mean(),
    "avg_frame_f1_car":           eval_df["f1_car"].mean(),
}])

summary_df.to_csv(OUTPUT_SUMMARY_CSV, index=False)

# =====================================================
# 12. Print summary
# =====================================================
print("Evaluation finished!")
print(f"Frame-by-frame evaluation saved to: {OUTPUT_EVAL_CSV}")
print(f"Summary CSV saved to:               {OUTPUT_SUMMARY_CSV}")

print("\n=== GLOBAL METRICS (ALL OBJECTS) ===")
print(f"TP: {global_tp}  |  FP: {global_fp}  |  FN: {global_fn}")
print(f"Precision: {gp:.4f}  |  Recall: {gr:.4f}  |  F1-score: {gf:.4f}")

print("\n=== GLOBAL METRICS (PERSON) ===")
print(f"TP: {global_tp_person}  |  FP: {global_fp_person}  |  FN: {global_fn_person}")
print(f"Precision: {gp_p:.4f}  |  Recall: {gr_p:.4f}  |  F1-score: {gf_p:.4f}")

print("\n=== GLOBAL METRICS (CAR) ===")
print(f"TP: {global_tp_car}  |  FP: {global_fp_car}  |  FN: {global_fn_car}")
print(f"Precision: {gp_c:.4f}  |  Recall: {gr_c:.4f}  |  F1-score: {gf_c:.4f}")

print("\n=== AVERAGE PER FRAME ===")
print(f"Avg Precision: {eval_df['precision'].mean():.4f}")
print(f"Avg Recall:    {eval_df['recall'].mean():.4f}")
print(f"Avg F1-score:  {eval_df['f1_score'].mean():.4f}")

print("\nSample rows:")
print(eval_df.head(10))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total GT boxes after filtering: 2557
GT class distribution:
type
Car           2272
Pedestrian     106
Van            102
Truck           77
Name: count, dtype: int64

Evaluation finished!
Frame-by-frame evaluation saved to: /content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI_results_presentation2/kitti_detection_eval_iou.csv
Summary CSV saved to:               /content/drive/MyDrive/Colab Notebooks/EPL445/FINAL PROJECT/KITTI_results_presentation2/kitti_detection_eval_summary.csv

=== GLOBAL METRICS (ALL OBJECTS) ===
TP: 1915  |  FP: 717  |  FN: 642
Precision: 0.7276  |  Recall: 0.7489  |  F1-score: 0.7381

=== GLOBAL METRICS (PERSON) ===
TP: 24  |  FP: 3  |  FN: 82
Precision: 0.8889  |  Recall: 0.2264  |  F1-score: 0.3609

=== GLOBAL METRICS (CAR) ===
TP: 1891  |  FP: 714  |  FN: 560
Precision: 0.7259  |  Recall: 0.7715  |  F1-score: 0.7480

==